In [3]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [9]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [10]:
import torch.nn as nn

class MySimpleNN(nn.Module):

    def __init__(self, num_features):

        super().__init__()

        self.linear = nn.Linear(num_features, 1)

    def forward(self, features):

        logits = self.linear(features)

        return logits

In [11]:
model = MySimpleNN(X_train.shape[1])

In [12]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

In [13]:
epochs = 40

for epoch in range(epochs):

    logits = model(X_train_tensor)

    loss = criterion(logits, y_train_tensor.unsqueeze(1).float())

    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    print(f'Epoch: {epoch+1}, Loss: {loss.item():.4f}')

Epoch: 1, Loss: 0.7649
Epoch: 2, Loss: 0.5730
Epoch: 3, Loss: 0.4715
Epoch: 4, Loss: 0.4095
Epoch: 5, Loss: 0.3672
Epoch: 6, Loss: 0.3361
Epoch: 7, Loss: 0.3121
Epoch: 8, Loss: 0.2928
Epoch: 9, Loss: 0.2768
Epoch: 10, Loss: 0.2634
Epoch: 11, Loss: 0.2519
Epoch: 12, Loss: 0.2418
Epoch: 13, Loss: 0.2329
Epoch: 14, Loss: 0.2250
Epoch: 15, Loss: 0.2179
Epoch: 16, Loss: 0.2115
Epoch: 17, Loss: 0.2057
Epoch: 18, Loss: 0.2004
Epoch: 19, Loss: 0.1955
Epoch: 20, Loss: 0.1909
Epoch: 21, Loss: 0.1867
Epoch: 22, Loss: 0.1828
Epoch: 23, Loss: 0.1792
Epoch: 24, Loss: 0.1758
Epoch: 25, Loss: 0.1726
Epoch: 26, Loss: 0.1696
Epoch: 27, Loss: 0.1668
Epoch: 28, Loss: 0.1641
Epoch: 29, Loss: 0.1615
Epoch: 30, Loss: 0.1591
Epoch: 31, Loss: 0.1569
Epoch: 32, Loss: 0.1547
Epoch: 33, Loss: 0.1526
Epoch: 34, Loss: 0.1507
Epoch: 35, Loss: 0.1488
Epoch: 36, Loss: 0.1470
Epoch: 37, Loss: 0.1452
Epoch: 38, Loss: 0.1436
Epoch: 39, Loss: 0.1420
Epoch: 40, Loss: 0.1405


In [14]:
with torch.no_grad():

    logits = model(X_test_tensor)

    predictions = torch.sigmoid(logits)

    predictions = (predictions > 0.5).float()

    accuracy = (predictions == y_test_tensor.unsqueeze(1).float()).float().mean()

    print(f'Accuracy: {accuracy.item():.4f}')

Accuracy: 0.9561


In [15]:
# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA Version: {torch.version.cuda}")

# Move model and tensors to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

GPU Available: True
GPU Name: Tesla T4
CUDA Version: 12.8
Using device: cuda
